In [1]:
# 首先設定路徑和基本導入
import os
import sys
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple
import glob
import shutil
import logging
import traceback
import numpy as np
import pandas as pd
import talib
from tqdm import tqdm

# 定義配置類別
@dataclass
class TWStockConfig:
    """台股數據分析核心配置"""
    
    # 基礎路徑配置
    base_dir: Path = Path("D:/Min/Python/Project/FA_Data")  # 修改為你的數據目錄
    
    # 數據目錄
    data_dir: Path = None
    daily_price_dir: Path = None 
    meta_data_dir: Path = None
    technical_dir: Path = None
    
    # 關鍵檔案路徑
    market_index_file: Path = None
    industry_index_file: Path = None
    stock_data_file: Path = None
    
    # 數據參數
    default_start_date: str = "2014-01-01"
    backup_keep_days: int = 7
    min_data_days: int = 30
    
    def __post_init__(self):
        """初始化衍生屬性"""
        # 設定數據目錄
        self.data_dir = self.base_dir
        self.daily_price_dir = self.data_dir / 'daily_price'
        self.meta_data_dir = self.data_dir / 'meta_data'
        self.technical_dir = self.data_dir / 'technical_analysis'
        
        # 設定關鍵檔案路徑
        self.market_index_file = self.meta_data_dir / 'market_index.csv'
        self.industry_index_file = self.meta_data_dir / 'industry_index.csv'
        self.stock_data_file = self.meta_data_dir / 'stock_data_whole.csv'
        
        # 確保所需目錄存在
        self._ensure_directories()
    
    def _ensure_directories(self):
        """確保所需目錄結構存在"""
        directories = [
            self.daily_price_dir,
            self.meta_data_dir,
            self.technical_dir,
        ]
        for directory in directories:
            directory.mkdir(parents=True, exist_ok=True)
            
    @property
    def backup_dir(self) -> Path:
        """備份目錄路徑"""
        return self.meta_data_dir / 'backup'
    
    def get_technical_file(self, stock_id: str) -> Path:
        """取得特定股票的技術分析檔案路徑"""
        return self.technical_dir / f'{stock_id}_indicators.csv'
    
    def get_daily_price_file(self, date: str) -> Path:
        """取得特定日期的價格檔案路徑"""
        return self.daily_price_dir / f'{date}.csv'

class MarketDateRange:
    """市場數據日期範圍控制"""
    def __init__(self, start_date: str = None, end_date: str = None):
        self.end_date = end_date if end_date else datetime.today().strftime('%Y-%m-%d')
        self.start_date = start_date
        
    @classmethod
    def last_n_days(cls, n: int) -> 'MarketDateRange':
        """創建最近 n 天的日期範圍"""
        end_date = datetime.today()
        start_date = end_date - timedelta(days=n)
        return cls(
            start_date=start_date.strftime('%Y-%m-%d'),
            end_date=end_date.strftime('%Y-%m-%d')
        )
    
    @classmethod
    def last_month(cls) -> 'MarketDateRange':
        """創建最近一個月的日期範圍"""
        return cls.last_n_days(30)
    
    @classmethod
    def last_quarter(cls) -> 'MarketDateRange':
        """創建最近一季的日期範圍"""
        return cls.last_n_days(90)
    
    @classmethod
    def last_year(cls) -> 'MarketDateRange':
        """創建最近一年的日期範圍"""
        return cls.last_n_days(365)
    
    @classmethod
    def year_to_date(cls) -> 'MarketDateRange':
        """創建今年至今的日期範圍"""
        return cls(
            start_date=datetime.today().replace(month=1, day=1).strftime('%Y-%m-%d')
        )
        
    @property
    def date_range_str(self) -> str:
        """返回日期範圍的字符串表示"""
        return f"從 {self.start_date or '最早'} 到 {self.end_date}"

# 設置日誌
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# 清除現有的處理器
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# 設定日誌處理器
file_handler = logging.FileHandler('technical_calculation.log', encoding='utf-8')
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
console_handler.setLevel(logging.ERROR)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

In [2]:
def setup_logging():
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)
    
    # 清除現有的處理器
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
    
    # 檔案處理器
    file_handler = logging.FileHandler('technical_calculation.log', encoding='utf-8')
    file_handler.setFormatter(
        logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    )
    
    # 控制台處理器
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(
        logging.Formatter('%(levelname)s: %(message)s')
    )
    console_handler.setLevel(logging.ERROR)
    
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    return logger

logger = setup_logging()

In [3]:
def process_price_data(df):
    """
    處理價格數據，包含資料清理和格式轉換
    """
    try:
        # 複製DataFrame避免修改原始資料
        df = df.copy()
        # 處理漲跌符號欄位中的 HTML 標籤
        if '漲跌(+/-)' in df.columns:
            replace_dict = {
                '<p style= color:red>+</p>': '+',
                '<p style= color:green>-</p>': '-',
                '<p style =color:red>+</p>': '+',
                '<p style =color:green>-</p>': '-'
            }
            df['漲跌(+/-)'] = df['漲跌(+/-)'].map(lambda x: replace_dict.get(x, x))
        price_columns = ['開盤價', '最高價', '最低價', '收盤價', '最後揭示買價', '最後揭示賣價']
        for col in price_columns:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: str(x).replace(',', '') if isinstance(x, str) else x)
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # 處理成交量欄位
        volume_columns = ['成交股數', '成交筆數', '成交金額']
        for col in volume_columns:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: str(x).replace(',', '') if isinstance(x, str) else x)
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # 處理日期欄位
        if '日期' in df.columns:
            df['日期'] = pd.to_datetime(df['日期']).dt.strftime('%Y-%m-%d')
            
        # 處理特殊字符
        df = df.replace('--', np.nan)
        df = df.replace('', np.nan)
        
        return df
        
    except Exception as e:
        logging.error(f"Error in process_price_data: {e}")
        logging.error(traceback.format_exc())
        return None

In [4]:
def validate_config():
    """
    驗證必要的目錄和檔案是否存在
    """
    base_path = Path("D:/Min/Python/Project/FA_Data")
    required_dirs = [
        "meta_data",
        "technical_analysis",
        "meta_data\\backup"
    ]
    
    try:
        for dir_path in required_dirs:
            full_path = os.path.join(base_path, dir_path)
            if not os.path.exists(full_path):
                os.makedirs(full_path)
                logging.info(f"Created directory: {full_path}")
                
        return True
    except Exception as e:
        logging.error(f"Error validating config: {e}")
        return False

In [5]:
def calculate_and_store_indicators(df, stock_id=None, date=None, by_date=False):
    """Calculate technical indicators and save to files"""
    try:
        # 數據預處理
        df = process_price_data(df)
        if df is None:
            return False
            
        # 檢查數據有效性
        price_columns = ['開盤價', '最高價', '最低價', '收盤價']
        if df[price_columns].isnull().all().any():
            logging.warning(f"Missing price data for {'date '+date if by_date else 'stock ID '+stock_id}")
            return False

        # 數據量檢查
        if len(df) < 30:
            logger.warning(f"股票 {stock_id} 資料量不足30天")
            return None
            
        # 排序並重設索引
        df = df.sort_values('日期').reset_index(drop=True)
        
        # 使用收盤價計算技術指標
        close_prices = df['收盤價'].values
        high_prices = df['最高價'].values
        low_prices = df['最低價'].values
        
        # 初始化所有技術指標欄位為np.nan
        for col in ['SMA30', 'DEMA30', 'EMA30', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 
                   'slowk', 'slowd', 'TSF', 'middleband', 'SAR']:
            df[col] = np.nan
        
        # 計算移動平均線系列
        df['SMA30'] = talib.SMA(close_prices, timeperiod=30)
        df['DEMA30'] = talib.DEMA(close_prices, timeperiod=30)
        df['EMA30'] = talib.EMA(close_prices, timeperiod=30)
        
        # 計算RSI - 修正版本
        try:
            # 直接使用原始的close_prices計算RSI
            rsi = talib.RSI(close_prices, timeperiod=14)
            # 將無限值和NaN替換為50（中性值）
            rsi = np.nan_to_num(rsi, nan=50.0, posinf=100.0, neginf=0.0)
            # 確保值在0-100範圍內
            df['RSI'] = np.clip(rsi, 0, 100)
        except Exception as e:
            logging.warning(f"Error calculating RSI for {stock_id}: {str(e)}")
            df['RSI'] = 50.0  # 錯誤時使用中性值
        
        # 計算MACD
        macd, signal, hist = talib.MACD(close_prices, 
                                      fastperiod=12, 
                                      slowperiod=26, 
                                      signalperiod=9)
        df['MACD'] = macd
        df['MACD_signal'] = signal
        df['MACD_hist'] = hist
        
        # 計算KD指標
        try:
            slowk, slowd = talib.STOCH(high_prices, 
                                     low_prices, 
                                     close_prices,
                                     fastk_period=5, 
                                     slowk_period=3, 
                                     slowk_matype=0,
                                     slowd_period=3, 
                                     slowd_matype=0)
            # 處理可能的無限值和NaN
            slowk = np.nan_to_num(slowk, nan=50.0, posinf=100.0, neginf=0.0)
            slowd = np.nan_to_num(slowd, nan=50.0, posinf=100.0, neginf=0.0)
            df['slowk'] = np.clip(slowk, 0, 100)
            df['slowd'] = np.clip(slowd, 0, 100)
        except Exception as e:
            logging.warning(f"Error calculating KD for {stock_id}: {str(e)}")
            df['slowk'] = 50.0
            df['slowd'] = 50.0
        
        # 計算TSF
        df['TSF'] = talib.TSF(close_prices, timeperiod=14)
        
        # 計算布林通道
        upper, middle, lower = talib.BBANDS(close_prices, 
                                          timeperiod=30, 
                                          nbdevup=2, 
                                          nbdevdn=2, 
                                          matype=0)
        df['middleband'] = middle
        
        # 計算SAR
        try:
            sar = talib.SAR(high_prices, 
                           low_prices, 
                           acceleration=0.02, 
                           maximum=0.2)
            # 設定合理的價格限制
            max_price = df['最高價'].max() * 1.5
            min_price = df['最低價'].min() * 0.5
            sar = np.nan_to_num(sar, nan=df['收盤價'].mean())
            df['SAR'] = np.clip(sar, min_price, max_price)
        except Exception as e:
            logging.warning(f"Error calculating SAR for {stock_id}: {str(e)}")
            df['SAR'] = df['收盤價']  # 使用收盤價作為預設值
        
        # 最後進行一次全面的數值檢查
        technical_columns = ['RSI', 'slowk', 'slowd']
        for col in technical_columns:
            if col in df.columns:
                df[col] = np.clip(df[col], 0, 100)
        
        # 儲存結果
        base_path = Path("D:/Min/Python/Project/FA_Data")
        save_path = base_path / "technical_analysis" / f"{stock_id}_indicators.csv"
        
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        
        return df
        
    except Exception as e:
        logging.error(f"Error processing stock {stock_id}: {e}")
        logging.error(traceback.format_exc())
        return None

In [6]:
def process_stock_data_batch():
    """批次處理股票資料"""
    try:
        base_path = "D:/Min/Python/Project/FA_Data"
        stock_data_whole = pd.read_csv(
            f"{base_path}/meta_data/stock_data_whole.csv",
            dtype={'證券代號': str, '證券名稱': str},
            low_memory=False
        )
        
        # 修改為只保留4位數股票代號
        stock_data_whole = stock_data_whole[
            stock_data_whole['證券代號'].str.match(r'^\d{4}$')
        ]
        
        all_processed_data = []
        grouped = stock_data_whole.groupby('證券代號')
        total_stocks = len(grouped)
        
        processed_count = 0
        failed_stocks = []
        
        # 使用tqdm包裝grouped和增加描述性信息
        progress_bar = tqdm(
            grouped, 
            total=total_stocks,
            desc="Processing stocks",
            unit="stock",
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
        )
        
        for stock_id, group_df in progress_bar:
            # 檢查資料量是否足夠
            if len(group_df) < 30:  # 最少需要30天數據
                failed_stocks.append(f"{stock_id}(資料不足)")
                continue
                
            result_df = calculate_and_store_indicators(group_df.copy(), stock_id=stock_id)
            if isinstance(result_df, pd.DataFrame):
                all_processed_data.append(result_df)
                processed_count += 1
                progress_bar.set_postfix({'Success': f"{processed_count}/{total_stocks}"})
            else:
                failed_stocks.append(stock_id)
                progress_bar.set_postfix({'Failed': stock_id})

        progress_bar.close()

        if all_processed_data:
            # 合併所有處理後的資料
            logging.info("Merging processed data...")
            all_stocks_df = pd.concat(all_processed_data, ignore_index=True)
            
            # 備份和儲存
            today = datetime.now().strftime('%Y%m%d')
            backup_dir = f"{base_path}\\meta_data\\backup"
            backup_path = f"{backup_dir}\\all_stocks_data_{today}.csv"
            
            # 保存新的合併檔案
            logging.info("Saving merged data...")
            all_stocks_df.to_csv(f"{base_path}\\meta_data\\all_stocks_data.csv", 
                               index=False, encoding='utf-8-sig')
            
            # 建立備份
            shutil.copy2(f"{base_path}\\meta_data\\all_stocks_data.csv", backup_path)
            
            # 清理舊備份（保留最近7天）
            backup_files = glob.glob(f"{backup_dir}\\all_stocks_data_*.csv")
            backup_files.sort(reverse=True)
            for old_backup in backup_files[7:]:
                os.remove(old_backup)
            
            # 輸出處理結果統計
            logging.info(f"Processing completed:")
            logging.info(f"- Successfully processed: {processed_count}/{total_stocks} stocks")
            if failed_stocks:
                logging.info(f"- Failed stocks: {', '.join(failed_stocks)}")
            
            return True
        else:
            logging.warning("No data was successfully processed")
            return False
            
    except Exception as e:
        logging.error(f"Error in batch processing: {e}")
        logging.error(traceback.format_exc())
        return False

In [7]:
def main():
    """主程序"""
    try:
        config = TWStockConfig()
        date_range = MarketDateRange.last_n_days(45)
        
        logging.info("Starting batch processing...")
        if process_stock_data_batch():
            logging.info("Batch processing completed successfully")
            return True
        else:
            logging.error("Batch processing failed")
            return False
            
    except Exception as e:
        logging.error(f"Error in main process: {e}")
        logging.error(traceback.format_exc())
        return False

In [8]:
if __name__ == "__main__":
    if validate_config():
        success = main()
        if success:
            logging.info("Process completed successfully")
            
            # 顯示一個股票的處理結果作為範例
            example_stock = "2330"
            try:
                df = pd.read_csv(f"D:\\Min\\Python\\Project\\FA_Data\\technical_analysis\\{example_stock}_indicators.csv")
                print(f"\n{example_stock} 技術指標計算結果範例:")
                print(df.tail())
            except Exception as e:
                logging.error(f"Error showing example results: {e}")
    else:
        logging.error("Configuration validation failed")

Processing stocks: 100%|████████████████████████████████████████████████████████| 1111/1111 [02:40<00:00,  6.93stock/s]



2330 技術指標計算結果範例:
      證券代號 證券名稱      成交股數   成交筆數         成交金額     開盤價     最高價     最低價     收盤價  \
2611  2330  台積電  30255281  25230  32409791931  1070.0  1075.0  1065.0  1075.0   
2612  2330  台積電  32359253  31326  34496134808  1070.0  1075.0  1060.0  1065.0   
2613  2330  台積電  26948940  47194  28278772503  1045.0  1060.0  1045.0  1045.0   
2614  2330  台積電  26574460  25096  28185087817  1060.0  1065.0  1055.0  1060.0   
2615  2330  台積電  24329785  25787  25873117992  1060.0  1070.0  1055.0  1065.0   

     漲跌(+/-)  ...        EMA30        RSI      MACD  MACD_signal  MACD_hist  \
2611       +  ...  1038.896692  59.678209  7.823844     2.159840   5.664004   
2612       -  ...  1040.580776  56.522270  8.551336     3.438139   5.113197   
2613       -  ...  1040.865887  50.742633  7.428415     4.236194   3.192221   
2614     NaN  ...  1042.100346  54.500437  7.660561     4.921067   2.739494   
2615       +  ...  1043.577743  55.713267  8.154002     5.567654   2.586348   

          slowk     